In [1]:
import sys
# Add parent directory to path to import helper_function module
sys.path.insert(0, '..')

import helper_functions

import eelbrain
from scipy.io import loadmat
from constants import *

## Create gammatone spectrogram

We start by using a gammatone filterbank which mimics the cochlea to create a 128 band gammatone spectrogram.

In [2]:
def compute_gammatone(stimulus):
    """Compute and save gammatone spectrogram for a single stimulus."""
    dst = STIMULUS_DIR / f'{stimulus}-gammatone.pickle'

    if dst.exists():
        return

    wav = eelbrain.load.wav(STIMULUS_DIR / f'{stimulus}.wav')
    gt  = eelbrain.gammatone_bank(
        wav,
        GAMMATONE_FREQUENCY_RANGE_LOW,
        GAMMATONE_FREQUENCY_RANGE_HIGH,
        GAMMATONE_SPECTROGRAM_BANDS,
        location='left',
        tstep=0.001
    )
    eelbrain.save.pickle(gt, dst)
    print(f"  ✓ Saved {dst}")


def compute_stimulus_predictors(stimulus):
    """Compute and save all predictors for a single stimulus."""
    dst_envelope = SELF_COMPUTED_DIR / f'{stimulus}~{PREDICTOR_TYPE.ENVELOPE.value}.pickle'

    if dst_envelope.exists():
        print(f"{stimulus}: predictors exist, skipping.")
        return

    gt     = eelbrain.load.unpickle(STIMULUS_DIR / f'{stimulus}-gammatone.pickle')
    gt_log = (gt + 1).log()
    gt_on  = eelbrain.edge_detector(gt_log, c=30)

    predictors = {
        PREDICTOR_TYPE.ENVELOPE:                gt_log.sum('frequency'),
        PREDICTOR_TYPE.ENVELOPE_ONSET:          gt_on.sum('frequency'),
        PREDICTOR_TYPE.SPECTROGRAM_8_BAND:      gt_log.bin(nbins=8, func='sum', dim='frequency'),
        PREDICTOR_TYPE.SPECTROGRAM_ONSET_8_BAND: gt_on.bin(nbins=8, func='sum', dim='frequency'),
    }

    for predictor_type, predictor in predictors.items():
        dst = SELF_COMPUTED_DIR / f'{stimulus}~{predictor_type.value}.pickle'
        eelbrain.save.pickle(helper_functions.process_predictor(predictor), dst)

    print(f"  ✓ Saved predictors for {stimulus}")

In [3]:
for stimulus in helper_functions.get_stimuli_paths():
    compute_gammatone(stimulus)
    compute_stimulus_predictors(stimulus)

print("Done computing stimulus predictors.")

marianne_story5_trial_5: predictors exist, skipping.
aske_story2_trial_3: predictors exist, skipping.
marianne_story4_trial_22: predictors exist, skipping.
marianne_story4_trial_23: predictors exist, skipping.
aske_story2_trial_2: predictors exist, skipping.
marianne_story5_trial_4: predictors exist, skipping.
marianne_story5_trial_6: predictors exist, skipping.
marianne_story4_trial_21: predictors exist, skipping.
aske_story2_trial_11: predictors exist, skipping.
aske_story2_trial_10: predictors exist, skipping.
marianne_story4_trial_20: predictors exist, skipping.
aske_story2_trial_1: predictors exist, skipping.
marianne_story5_trial_7: predictors exist, skipping.
marianne_story5_trial_3: predictors exist, skipping.
aske_story2_trial_5: predictors exist, skipping.
aske_story3_trial_13: predictors exist, skipping.
marianne_story4_trial_24: predictors exist, skipping.
marianne_story4_trial_18: predictors exist, skipping.
aske_story5_trial_9: predictors exist, skipping.
aske_story5_tria

# Match predictors to trials and concatenate


In [4]:
#helper_functions.get_trials()
# Trials dictionary of type 
# trials = { 
#     1: { 
#         'attended': 'marianne_story3_trial_1', 
#         'ignored': 'aske_story4_trial_1'}
#     }
# }

In [5]:
def concatenate_predictors(predictor_names, attention: ATTENTION_TYPE, padded=False):
    """
    Concatenate predictors across trials for a single attention type.

    Args:
        predictor_names: A single PREDICTOR_TYPE or list of PREDICTOR_TYPE values.
        attention:       ATTENTION_TYPE.ATTENDED or ATTENTION_TYPE.IGNORED.
        padded:          Whether to pad each predictor before concatenating.

    Returns:
        The concatenated eelbrain object.
    """
    trials = helper_functions.get_trials()

    # Normalize to list
    if isinstance(predictor_names, PREDICTOR_TYPE):
        predictor_names = [predictor_names]

    concat_preds = []

    for trial_id, trial in trials.items():
        story = trial[attention.value]  # e.g. trial['attended_story'] or trial['ignored_story']

        # Load and combine all requested predictors for this trial
        preds_for_trial = []
        for predictor_name in predictor_names:
            pred = eelbrain.load.unpickle(
                SELF_COMPUTED_DIR / f'{story}~{predictor_name.value}.pickle'
            )
            preds_for_trial.append(pred)

        # If multiple predictors, combine them (e.g. as a NDVar with multiple cases)
        trial_pred = eelbrain.combine(preds_for_trial) if len(preds_for_trial) > 1 else preds_for_trial[0]

        if padded:
            trial_pred = eelbrain.pad(
                trial_pred,
                tstart=-PADDING_ONSET,
                tstop=trial_pred.time.tstop + PADDING_OFFSET,
            )

        concat_preds.append(trial_pred)

    name = helper_functions.get_predictor_name(predictor_names, attention, padded)
    concat = eelbrain.concatenate(concat_preds, name=name)

    out_path = SELF_COMPUTED_CONCAT_DIR / f'{name}_concat.pickle'
    eelbrain.save.pickle(concat, out_path)
    print(f"Saved {attention.value} → {out_path}")

    return concat

In [6]:
checks = [
    (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.ATTENDED, False),
    (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.IGNORED,  False),
    (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.ATTENDED, True),
    (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.IGNORED,  True),
    (PREDICTOR_TYPE.ENVELOPE_ONSET, ATTENTION_TYPE.ATTENDED, False),
    (PREDICTOR_TYPE.ENVELOPE_ONSET, ATTENTION_TYPE.IGNORED,  False),
]

for predictor, attention, padded in checks:
    concatenate_predictors(predictor, attention, padded)

print("Done concatenating predictors.")

Loaded 60 trials
Saved attended → /Users/sylvestereley/Data/cocoha4/predictors/concatenated/self_computed/attended_envelope_concat.pickle
Loaded 60 trials
Saved ignored → /Users/sylvestereley/Data/cocoha4/predictors/concatenated/self_computed/ignored_envelope_concat.pickle
Loaded 60 trials
Saved attended → /Users/sylvestereley/Data/cocoha4/predictors/concatenated/self_computed/attended_envelope_padded_concat.pickle
Loaded 60 trials
Saved ignored → /Users/sylvestereley/Data/cocoha4/predictors/concatenated/self_computed/ignored_envelope_padded_concat.pickle
Loaded 60 trials
Saved attended → /Users/sylvestereley/Data/cocoha4/predictors/concatenated/self_computed/attended_envelope_onset_concat.pickle
Loaded 60 trials
Saved ignored → /Users/sylvestereley/Data/cocoha4/predictors/concatenated/self_computed/ignored_envelope_onset_concat.pickle
Done concatenating predictors.


In [7]:
# SANITY CHECK: Check dimensions of concatenated predictors
checks = [
    (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.ATTENDED, False),
    (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.IGNORED,  False),
    (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.ATTENDED, True),
    (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.IGNORED,  True),
    (PREDICTOR_TYPE.ENVELOPE_ONSET, ATTENTION_TYPE.ATTENDED, False),
    (PREDICTOR_TYPE.ENVELOPE_ONSET, ATTENTION_TYPE.IGNORED,  False),
]

for predictor, attention, padded in checks:
    name = helper_functions.get_predictor_name(predictor, attention, padded)
    print(eelbrain.load.unpickle(SELF_COMPUTED_CONCAT_DIR / f'{name}_concat.pickle'))


<NDVar 'attended_envelope': 192000 time>
<NDVar 'ignored_envelope': 192000 time>
<NDVar 'attended_envelope_padded': 196260 time>
<NDVar 'ignored_envelope_padded': 196260 time>
<NDVar 'attended_envelope_onset': 192000 time>
<NDVar 'ignored_envelope_onset': 192000 time>
